# Member 3: Prompt Engineering, Decoding, and Evaluation

This notebook runs only the work assigned to Member 3:

- **E2:** zero-shot, few-shot, role/context, and constraint-based prompt comparison
- **E3:** greedy, temperature, top-k, and top-p decoding comparison
- **Evaluation:** per-output metrics and an aggregate comparison matrix

It reuses Member 1's prepared dataset and shared model loader so all experiments use the same inputs and model as the baseline.

## 1. Colab runtime setup

Before continuing, select **Runtime > Change runtime type > T4 GPU**. The quantized model loader requires an NVIDIA GPU.

In [13]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/2025paml587-byte/BITS_PAML_AGAI_G10_A1.git"
REPO_DIR = Path("/content/BITS_PAML_AGAI_G10_A1")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")

Working directory: /content/BITS_PAML_AGAI_G10_A1


In [14]:
# Runtime packages needed by the shared model loader and Member 3 module.
%pip install -q datasets pandas numpy python-dotenv transformers accelerate bitsandbytes sentencepiece

In [15]:
import torch

assert torch.cuda.is_available(), (
    "CUDA GPU not detected. In Colab, select Runtime > Change runtime type > T4 GPU."
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")

GPU: Tesla T4
PyTorch: 2.11.0+cu130


## 2. Import and test the Member 3 utilities

This lightweight check validates prompt rendering and evaluation logic before loading the LLM.

In [16]:
from src.member3_experiments import (
    DECODING_CONFIGS,
    PROMPT_VARIANTS,
    build_evaluation_matrix,
    build_prompt,
    load_experiment_samples,
    run_decoding_study,
    run_prompt_comparison,
    save_results,
    set_reproducible_seed,
)

assert len(PROMPT_VARIANTS) == 4
assert len(DECODING_CONFIGS) == 4
test_prompt = build_prompt("zero_shot", "Alex: The meeting is at 3 PM.")
assert "Alex: The meeting is at 3 PM." in test_prompt
print("Member 3 utility checks passed.")
print("Prompt variants:", list(PROMPT_VARIANTS))
print("Decoding configurations:", list(DECODING_CONFIGS))

Member 3 utility checks passed.
Prompt variants: ['zero_shot', 'few_shot', 'role_context', 'constraint_based']
Decoding configurations: ['greedy', 'temperature', 'top_k', 'top_p']


## 3. Load the controlled experiment samples

Three examples satisfy the assignment's minimum evidence requirement. Increase `SAMPLE_LIMIT` after the initial test if more evidence is needed.

In [17]:
SAMPLE_LIMIT = 3
SAMPLING_REPEATS = 3
MAX_NEW_TOKENS = 150
SEED = 42

set_reproducible_seed(SEED)
samples = load_experiment_samples(
    data_path="data/experimental_subset.csv",
    sample_limit=SAMPLE_LIMIT,
)
display(samples[["id", "dialogue", "summary"]])

,id,dialogue,summary
0,13811908,violet hi i came across this austins article a...,violet sent claire austins article
1,13731387,mel right its time i bought some christmas pre...,mel bought christmas presents while evie keeps...
2,13727910,carol can you pls pick up when i call you caro...,fiona cant talk to carol because shes very bus...


## 4. Load the shared quantized model

This uses the repository's shared model configuration rather than introducing a different model for Member 3. The first run downloads the model and may take several minutes.

In [18]:
from src.model_pipeline import load_quantized_model

model, tokenizer = load_quantized_model()
print("Shared model and tokenizer loaded successfully.")

📦 Initializing configuration for model: microsoft/Phi-3-mini-4k-instruct...
⏳ Loading Tokenizer...
⏳ Loading Quantized Causal Language Model into memory...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ Model and Tokenizer successfully loaded!
Shared model and tokenizer loaded successfully.


## 5. Run E2: four prompt variants

Greedy decoding is fixed for every prompt so the prompt itself is the only experimental variable.

In [19]:
e2_records = run_prompt_comparison(
    model=model,
    tokenizer=tokenizer,
    samples=samples,
    max_new_tokens=MAX_NEW_TOKENS,
)
print(f"E2 completed: {len(e2_records)} outputs generated.")

AttributeError: 'DynamicCache' object has no attribute 'seen_tokens'

## 6. Run E3: decoding-control study

The constraint-based prompt is fixed for all runs. Sampling configurations are repeated to measure variability; deterministic greedy decoding runs once per sample.

In [ ]:
e3_records = run_decoding_study(
    model=model,
    tokenizer=tokenizer,
    samples=samples,
    prompt_variant="constraint_based",
    max_new_tokens=MAX_NEW_TOKENS,
    sampling_repeats=SAMPLING_REPEATS,
)
print(f"E3 completed: {len(e3_records)} outputs generated.")

## 7. Build and inspect the evaluation matrix

In [ ]:
all_records = e2_records + e3_records
evaluation_details, comparison_matrix = build_evaluation_matrix(all_records)

display(
    comparison_matrix.sort_values(
        ["experiment", "rouge_1_f1"], ascending=[True, False]
    )
)

In [ ]:
# Report-ready output examples for qualitative comparison.
display(
    evaluation_details[[
        "experiment",
        "variant",
        "trial",
        "sample_id",
        "generated_summary",
        "reference_summary",
        "rouge_1_f1",
        "rouge_l_f1",
        "source_token_support",
    ]].head(20)
)

### Metric interpretation

- **ROUGE-1 / ROUGE-L:** lexical agreement with the reference summary
- **Source-token support:** proportion of generated tokens also present in the dialogue; this is a lexical grounding proxy, not a complete factuality measure
- **Compression ratio:** generated-summary length divided by dialogue length
- **Output diversity:** fraction of unique repeated outputs for the same sample and configuration
- **Latency:** generation time in seconds

Use the displayed examples for qualitative analysis alongside these automated measurements.

## 8. Save and download Member 3 evidence

In [ ]:
result_paths = save_results(
    all_records, output_dir="data/member3_results"
)
result_paths

In [ ]:
# Optional: download the three CSV evidence files to your computer.
from google.colab import files

for result_path in result_paths.values():
    files.download(result_path)